# Jam 3-6: Model Prediksi Kemacetan (Random Forest)

Notebook ini digunakan untuk menghasilkan data kemacetan sintetis berdasarkan tabel `congestion_multipliers` dan melatih model `RandomForestClassifier` untuk memprediksi tingkat kemacetan.

In [ ]:
!pip install supabase pandas scikit-learn python-dotenv

In [ ]:
import os
import pandas as pd
import numpy as np
from supabase import create_client, Client
from dotenv import load_dotenv
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

load_dotenv(dotenv_path='../.env')

url: str = os.environ.get("SUPABASE_URL")
key: str = os.environ.get("SUPABASE_ANON_KEY")
supabase: Client = create_client(url, key)

### 1. Fetch Data dari Supabase

In [ ]:
response_segments = supabase.table("road_segments").select("*").execute()
response_multipliers = supabase.table("congestion_multipliers").select("*").execute()

df_segments = pd.DataFrame(response_segments.data)
df_multipliers = pd.DataFrame(response_multipliers.data)

df_segments.head()

### 2. Generate Synthetic Data
Kita akan membuat data latih sintetis berdasarkan base multiplier dari Supabase.

In [ ]:
synthetic_data = []
np.random.seed(42)

for index, row in df_multipliers.iterrows():
    segment_id = row['segment_id']
    day_type = row['day_type']
    start_hour = row['hour_start']
    end_hour = row['hour_end']
    base_multiplier = row['multiplier']
    
    # Ambil corridor_type
    corridor = df_segments[df_segments['id'] == segment_id]['corridor_type'].values[0]
    
    # Generate 50 baris data sintetis per kombinasi jam
    for hour in range(start_hour, end_hour + 1):
        for _ in range(50):
            # Tambahkan noise +/- 10%
            noise = np.random.uniform(-0.1, 0.1)
            multiplier = base_multiplier * (1 + noise)
            
            # Tentukan class/level
            if multiplier >= 2.0:
                level = 'high'
            elif multiplier >= 1.5:
                level = 'medium'
            else:
                level = 'low'
                
            synthetic_data.append({
                'segment_id': segment_id,
                'corridor_type': corridor,
                'day_type': day_type,
                'hour': hour,
                'is_peak': 1 if level == 'high' else 0,
                'congestion_level': level
            })

df_train = pd.DataFrame(synthetic_data)
print(f"Total synthetic data generated: {len(df_train)}")
df_train['congestion_level'].value_counts()

### 3. Feature Engineering & One-Hot Encoding

In [ ]:
# Kita hapus segment_id dari fitur numerik karena itu identifier
features = df_train[['hour', 'day_type', 'corridor_type', 'is_peak']]
target = df_train['congestion_level']

# One hot encoding untuk kategorikal
X = pd.get_dummies(features, columns=['day_type', 'corridor_type'])
y = target

# Simpan nama kolom untuk referensi prediksi di API nantinya
feature_columns = list(X.columns)
print("Fitur input: ", feature_columns)

### 4. Train Random Forest Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

### 5. Export Model (.joblib)

In [ ]:
# Export model beserta list fiturnya agar FastAPI tahu struktur input yang benar
model_package = {
    'model': rf_model,
    'feature_columns': feature_columns
}

joblib.dump(model_package, '../rf_model.joblib')
print("Model berhasil diexport ke backend/rf_model.joblib!")